# Train TRRUST Classifiers

Pipeline:

1. **Load** gene embeddings + TRRUST data.
2. **(Optional)** Restrict training data to a gene subset from a file.
3. **(Optional)** Write the current gene universe to a file for cross-model comparisons.
4. **Hyperparameter sweep with 5-fold CV per config** — 5 LRs × 4 epoch counts
   for binary + ternary. For each config we run stratified 5-fold CV and save
   per-fold train/test loss curves, per-fold classification reports, a summary
   CSV with mean/std accuracy and macro F1, and a heatmap of **mean macro F1**
   across LR × epochs under `REPORTS_DIR`.
5. **Final results** — re-run 5-fold CV at the chosen LR / epoch count for
   binary + ternary (inspect the sweep heatmap, then set `*_LR` / `*_EPOCHS`).
6. **Save final results** as a pickle for downstream analysis.

This notebook is model-agnostic — point `EMBEDDINGS_PATH` at any h5ad file produced
by `src/scgpt/encode.py` or `src/geneformer/encode.py`.


In [1]:
import itertools
import pickle
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report

from src.trrust import (
    BINARY_LABEL_NAMES,
    BINARY_LABELS,
    TERNARY_LABEL_NAMES,
    TERNARY_LABELS,
    count_relationship_pairs,
    cross_validate,
    filter_data_by_genes,
    generate_shared_none_pairs,
    load_binary_trrust_data,
    load_gene_embeddings,
    load_ternary_trrust_data,
)


## Configuration

Change `EMBEDDINGS_PATH` to swap foundation models, and point `REPORTS_DIR` /
`CV_RESULTS_PATH` at model-specific output locations. Set `GENE_SUBSET_PATH` to
restrict training to pairs where both TF and target appear in a gene-list file
(see `data/scgpt_binary_genes.txt`).

`LR_GRID` and `EPOCH_GRID` define the hyperparameter sweep; each `(lr, epochs)`
config is evaluated with 5-fold stratified CV. After inspecting the mean macro
F1 heatmap, fill in `BINARY_LR` / `BINARY_EPOCHS` / `TERNARY_LR` /
`TERNARY_EPOCHS` before running the final-results cells.


In [11]:
EMBEDDINGS_PATH = Path("../data/embeddings/geneformer_cd20.h5ad")
TRRUST_PATH = Path("../data/trrust_rawdata.human.tsv")
REPORTS_DIR = Path("../reports/geneformer/hp_tuning_cd20")
CV_RESULTS_PATH = Path("../data/results/geneformer_cv_results_cd20.pkl")
GENE_SUBSET_PATH: Path | None = None  # e.g. Path("../data/scgpt_binary_genes.txt") or None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

LR_GRID = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
EPOCH_GRID = [8, 12, 20, 50]


print(f"Device: {DEVICE}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"TRRUST: {TRRUST_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"CV results: {CV_RESULTS_PATH}")
print(f"Gene subset: {GENE_SUBSET_PATH}")


Device: cuda
Embeddings: ../data/embeddings/geneformer_cd20.h5ad
TRRUST: ../data/trrust_rawdata.human.tsv
Reports dir: ../reports/geneformer/hp_tuning_cd20
CV results: ../data/results/geneformer_cv_results_cd20.pkl
Gene subset: None


## Load embeddings and TRRUST data

In [12]:
gene_embeddings = load_gene_embeddings(EMBEDDINGS_PATH)
embsize = next(iter(gene_embeddings.values())).shape[0]

# Shared vocabulary across scFMs for this cell type. All training records
# (relationship + None) are restricted to this intersection so every scFM sees
# the same (tf, target) samples — cross-model agreement analysis then compares
# predictions on an identical pair set.
SHARED_SCFMS = ["scgpt", "geneformer"]
_scfm, _cell_tag = EMBEDDINGS_PATH.stem.split("_", 1)
scfm_vocabs = [
    set(load_gene_embeddings(EMBEDDINGS_PATH.parent / f"{name}_{_cell_tag}.h5ad").keys())
    for name in SHARED_SCFMS
]
shared_vocab = set.intersection(*scfm_vocabs)

n_binary_none = count_relationship_pairs(TRRUST_PATH, shared_vocab)
n_ternary_none = count_relationship_pairs(TRRUST_PATH, shared_vocab, exclude_unknown=True) // 2
binary_none_pairs = generate_shared_none_pairs(
    TRRUST_PATH, scfm_vocabs, n=n_binary_none, seed=42,
)
ternary_none_pairs = generate_shared_none_pairs(
    TRRUST_PATH, scfm_vocabs, n=n_ternary_none, seed=42,
)

binary_data = load_binary_trrust_data(
    TRRUST_PATH, gene_embeddings, none_pairs=binary_none_pairs,
)
ternary_data = load_ternary_trrust_data(
    TRRUST_PATH, gene_embeddings, none_pairs=ternary_none_pairs,
)

# Restrict relationship records to the shared vocab (None records are already
# drawn from the intersection, so they're unaffected).
binary_data = filter_data_by_genes(binary_data, shared_vocab)
ternary_data = filter_data_by_genes(ternary_data, shared_vocab)

print(f"Gene embeddings: {len(gene_embeddings)} genes, {embsize}d")
print(f"Shared vocab (intersection of {SHARED_SCFMS}): {len(shared_vocab)} genes")
print(f"Binary samples: {len(binary_data.records)} ({n_binary_none} shared None)")
print(f"Ternary samples: {len(ternary_data.records)} ({n_ternary_none} shared None)")


Gene embeddings: 11539 genes, 1152d
Shared vocab (intersection of ['scgpt', 'geneformer']): 10706 genes
Binary samples: 8076 (4038 shared None)
Ternary samples: 3135 (1045 shared None)


## (Optional) Restrict training data to a gene subset

If `GENE_SUBSET_PATH` is set, keep only TRRUST records where both TF and target are
listed in the file (one gene symbol per line). A no-op when `GENE_SUBSET_PATH is None`.


In [5]:
if GENE_SUBSET_PATH is not None:
    subset_genes = {
        line.strip()
        for line in Path(GENE_SUBSET_PATH).read_text().splitlines()
        if line.strip()
    }
    print(f"Subset file: {GENE_SUBSET_PATH} ({len(subset_genes)} genes)")

    before_binary = len(binary_data.records)
    before_ternary = len(ternary_data.records)
    binary_data = filter_data_by_genes(binary_data, subset_genes)
    ternary_data = filter_data_by_genes(ternary_data, subset_genes)
    print(f"Binary records: {before_binary} -> {len(binary_data.records)}")
    print(f"Ternary records: {before_ternary} -> {len(ternary_data.records)}")
else:
    print("GENE_SUBSET_PATH is None — using all records.")


Subset file: ../data/scgpt_binary_genes_nkt.txt (1782 genes)
Binary records: 9030 -> 7822
Ternary records: 3516 -> 3055


## (Optional) Save current gene lists for future cross-model runs

Write the union of TFs + targets from the current `binary_data` / `ternary_data` to
two `.txt` files so another run (e.g. a different embedding model) can be restricted
to the same gene universe via `GENE_SUBSET_PATH` above. Edit the output paths for
the model you are running.


In [8]:
BINARY_GENE_FILE = Path("../data/scgpt_binary_genes_nkt.txt")
TERNARY_GENE_FILE = Path("../data/scgpt_ternary_genes_nkt.txt")

binary_genes = sorted({g for r in binary_data.records for g in (r.tf, r.target)})
BINARY_GENE_FILE.write_text("\n".join(binary_genes) + "\n")
print(f"Saved {len(binary_genes)} binary genes to {BINARY_GENE_FILE}")

ternary_genes = sorted({g for r in ternary_data.records for g in (r.tf, r.target)})
TERNARY_GENE_FILE.write_text("\n".join(ternary_genes) + "\n")
print(f"Saved {len(ternary_genes)} ternary genes to {TERNARY_GENE_FILE}")


Saved 1802 binary genes to ../data/scgpt_binary_genes_nkt.txt
Saved 1590 ternary genes to ../data/scgpt_ternary_genes_nkt.txt


## Hyperparameter sweep helpers

Each config runs 5-fold stratified cross-validation (seed=42) so rankings are
robust to any single split. For every `(lr, epochs)` combination we save a
single plot with overlaid per-fold train/test loss curves, a text report with
per-fold classification reports plus mean ± std accuracy / macro F1, and a row
in `summary.csv`. After the loop, a heatmap of **mean macro F1** over LR ×
epochs is saved alongside the per-config outputs.


In [13]:
def _save_loss_plot(cv_result, lr, epochs, title_prefix, out_path):
    plt.figure(figsize=(8, 4))
    xs = range(1, epochs + 1)
    colors = plt.get_cmap("tab10").colors
    for i, fold in enumerate(cv_result.per_fold):
        color = colors[i % len(colors)]
        plt.plot(xs, fold.train_losses, color=color, linestyle="-",
                 label=f"Fold {i + 1} train")
        plt.plot(xs, fold.test_losses, color=color, linestyle="--",
                 label=f"Fold {i + 1} test")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} — lr={lr:.0e}, epochs={epochs}")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def _fold_macro_f1s(cv_result):
    return np.array([
        fold.classification_report["macro avg"]["f1-score"]
        for fold in cv_result.per_fold
    ])


def _save_text_report(cv_result, label_names, out_path):
    target_names = [label_names[i] for i in range(len(label_names))]
    name_to_label = {name: idx for idx, name in label_names.items()}
    lines = []
    for i, fold in enumerate(cv_result.per_fold):
        preds = fold.gene_predictions
        true_ids = preds["true_relationship"].map(name_to_label).to_numpy()
        pred_ids = preds["predicted_relationship"].map(name_to_label).to_numpy()
        lines.append(f"=== Fold {i + 1} ===")
        lines.append(classification_report(
            true_ids, pred_ids, target_names=target_names, zero_division=0
        ))
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    lines.append(f"Mean accuracy: {accs.mean():.3f} ± {accs.std():.3f}")
    lines.append(f"Mean macro F1: {f1s.mean():.3f} ± {f1s.std():.3f}")
    out_path.write_text("\n".join(lines))


def _summary_row(lr, epochs, cv_result, label_names):
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    row = {
        "lr": lr,
        "epochs": epochs,
        "mean_accuracy": accs.mean(),
        "std_accuracy": accs.std(),
        "mean_macro_f1": f1s.mean(),
        "std_macro_f1": f1s.std(),
    }
    agg = cv_result.aggregate_classification_report
    row["pooled_accuracy"] = agg["accuracy"]
    for name in ("macro avg", "weighted avg"):
        for metric in ("precision", "recall", "f1-score"):
            row[f"pooled_{name}_{metric}"] = agg[name][metric]
    for i in range(len(label_names)):
        name = label_names[i]
        row[f"pooled_{name}_precision"] = agg[name]["precision"]
        row[f"pooled_{name}_recall"] = agg[name]["recall"]
        row[f"pooled_{name}_f1"] = agg[name]["f1-score"]
    return row


def _save_f1_heatmap(summary_df, title, out_path):
    pivot = summary_df.pivot(
        index="lr", columns="epochs", values="mean_macro_f1"
    ).sort_index(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.0e}" for v in pivot.index])
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Learning rate")
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center",
                    color="white" if pivot.values[i, j] < pivot.values.mean() else "black")
    fig.colorbar(im, ax=ax, label="mean macro F1")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def run_hp_sweep(
    data,
    *,
    label_map,
    label_names,
    use_class_weights,
    out_dir,
    title_prefix,
    n_splits=5,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for lr, epochs in itertools.product(LR_GRID, EPOCH_GRID):
        cv_result = cross_validate(
            data,
            embsize=embsize,
            label_map=label_map,
            lr=lr,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            use_class_weights=use_class_weights,
            n_splits=n_splits,
            device=DEVICE,
            seed=42,
        )
        tag = f"lr{lr:.0e}_e{epochs}"
        _save_loss_plot(cv_result, lr, epochs, title_prefix, out_dir / f"loss_{tag}.png")
        _save_text_report(cv_result, label_names, out_dir / f"report_{tag}.txt")
        row = _summary_row(lr, epochs, cv_result, label_names)
        rows.append(row)
        print(f"  {tag}: mean acc={row['mean_accuracy']:.3f} ± {row['std_accuracy']:.3f}, "
              f"mean macro F1={row['mean_macro_f1']:.3f} ± {row['std_macro_f1']:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "summary.csv", index=False)
    _save_f1_heatmap(
        summary,
        f"{title_prefix} mean macro F1 (5-fold CV)",
        out_dir / "f1_heatmap.png",
    )
    return summary


## Hyperparameter sweep — binary classifier

20 configs (5 LRs × 4 epoch counts), each run through 5-fold stratified CV.
No class weights (binary data is balanced by construction). Outputs land in
`REPORTS_DIR/binary/`.


In [14]:
binary_summary = run_hp_sweep(
    binary_data,
    label_map=BINARY_LABELS,
    label_names=BINARY_LABEL_NAMES,
    use_class_weights=False,
    out_dir=REPORTS_DIR / "binary",
    title_prefix="Binary",
)
binary_summary.sort_values("mean_macro_f1", ascending=False).head()


  lr1e-02_e8: mean acc=0.730 ± 0.010, mean macro F1=0.729 ± 0.011
  lr1e-02_e12: mean acc=0.726 ± 0.008, mean macro F1=0.726 ± 0.008
  lr1e-02_e20: mean acc=0.721 ± 0.003, mean macro F1=0.721 ± 0.004
  lr1e-02_e50: mean acc=0.728 ± 0.004, mean macro F1=0.728 ± 0.004
  lr1e-03_e8: mean acc=0.740 ± 0.007, mean macro F1=0.740 ± 0.007
  lr1e-03_e12: mean acc=0.735 ± 0.006, mean macro F1=0.735 ± 0.006
  lr1e-03_e20: mean acc=0.734 ± 0.008, mean macro F1=0.734 ± 0.008
  lr1e-03_e50: mean acc=0.735 ± 0.009, mean macro F1=0.735 ± 0.009
  lr1e-04_e8: mean acc=0.736 ± 0.009, mean macro F1=0.735 ± 0.009
  lr1e-04_e12: mean acc=0.743 ± 0.010, mean macro F1=0.743 ± 0.011
  lr1e-04_e20: mean acc=0.748 ± 0.012, mean macro F1=0.748 ± 0.012
  lr1e-04_e50: mean acc=0.745 ± 0.011, mean macro F1=0.745 ± 0.012
  lr1e-05_e8: mean acc=0.646 ± 0.015, mean macro F1=0.646 ± 0.015
  lr1e-05_e12: mean acc=0.668 ± 0.014, mean macro F1=0.667 ± 0.014
  lr1e-05_e20: mean acc=0.693 ± 0.014, mean macro F1=0.693 ± 0.014

,lr,epochs,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,pooled_accuracy,pooled_macro avg_precision,pooled_macro avg_recall,pooled_macro avg_f1-score,pooled_weighted avg_precision,pooled_weighted avg_recall,pooled_weighted avg_f1-score,pooled_None_precision,pooled_None_recall,pooled_None_f1,pooled_Relationship_precision,pooled_Relationship_recall,pooled_Relationship_f1
10,0.0001,20,0.748392,0.011900,0.748347,0.011927,0.748390,0.748393,0.748390,0.748390,0.748393,0.748390,0.748390,0.747532,0.750124,0.748826,0.749254,0.746657,0.747953
11,0.0001,50,0.745173,0.011488,0.745091,0.011568,0.745171,0.745184,0.745171,0.745167,0.745184,0.745171,0.745167,0.743363,0.748886,0.746114,0.747006,0.741456,0.744221
9,0.0001,12,0.742696,0.010482,0.742658,0.010512,0.742694,0.742702,0.742694,0.742692,0.742702,0.742694,0.742692,0.741379,0.745419,0.743393,0.744024,0.739970,0.741992
4,0.0010,8,0.739846,0.006926,0.739785,0.006921,0.739846,0.739991,0.739846,0.739807,0.739991,0.739846,0.739807,0.745875,0.727588,0.736618,0.734107,0.752105,0.742997
8,0.0001,8,0.735514,0.009089,0.735476,0.009115,0.735513,0.735522,0.735513,0.735510,0.735522,0.735513,0.735510,0.734006,0.738732,0.736361,0.737039,0.732293,0.734658


## Hyperparameter sweep — ternary classifier

20 configs, each run through 5-fold stratified CV with inverse-frequency class
weights (ternary data is imbalanced). Outputs land in `REPORTS_DIR/ternary/`.


In [15]:
ternary_summary = run_hp_sweep(
    ternary_data,
    label_map=TERNARY_LABELS,
    label_names=TERNARY_LABEL_NAMES,
    use_class_weights=True,
    out_dir=REPORTS_DIR / "ternary",
    title_prefix="Ternary",
)
ternary_summary.sort_values("mean_macro_f1", ascending=False).head()


  lr1e-02_e8: mean acc=0.515 ± 0.016, mean macro F1=0.507 ± 0.017
  lr1e-02_e12: mean acc=0.508 ± 0.020, mean macro F1=0.499 ± 0.023
  lr1e-02_e20: mean acc=0.509 ± 0.017, mean macro F1=0.500 ± 0.021
  lr1e-02_e50: mean acc=0.515 ± 0.021, mean macro F1=0.506 ± 0.025
  lr1e-03_e8: mean acc=0.531 ± 0.029, mean macro F1=0.522 ± 0.030
  lr1e-03_e12: mean acc=0.537 ± 0.030, mean macro F1=0.526 ± 0.030
  lr1e-03_e20: mean acc=0.538 ± 0.031, mean macro F1=0.527 ± 0.031
  lr1e-03_e50: mean acc=0.540 ± 0.027, mean macro F1=0.527 ± 0.027
  lr1e-04_e8: mean acc=0.517 ± 0.020, mean macro F1=0.503 ± 0.019
  lr1e-04_e12: mean acc=0.529 ± 0.022, mean macro F1=0.514 ± 0.020
  lr1e-04_e20: mean acc=0.533 ± 0.025, mean macro F1=0.517 ± 0.023
  lr1e-04_e50: mean acc=0.537 ± 0.018, mean macro F1=0.518 ± 0.018
  lr1e-05_e8: mean acc=0.421 ± 0.019, mean macro F1=0.412 ± 0.018
  lr1e-05_e12: mean acc=0.437 ± 0.021, mean macro F1=0.428 ± 0.021
  lr1e-05_e20: mean acc=0.463 ± 0.022, mean macro F1=0.453 ± 0.021

,lr,epochs,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,pooled_accuracy,pooled_macro avg_precision,pooled_macro avg_recall,pooled_macro avg_f1-score,...,pooled_weighted avg_f1-score,pooled_Activation_precision,pooled_Activation_recall,pooled_Activation_f1,pooled_Repression_precision,pooled_Repression_recall,pooled_Repression_f1,pooled_None_precision,pooled_None_recall,pooled_None_f1
7,0.0010,50,0.540351,0.027093,0.527277,0.026772,0.540351,0.527822,0.527667,0.527726,...,0.539925,0.575686,0.582540,0.579093,0.404908,0.397590,0.401216,0.602871,0.602871,0.602871
6,0.0010,20,0.538118,0.031328,0.527272,0.031099,0.538118,0.527931,0.527383,0.527628,...,0.538589,0.573238,0.574603,0.573920,0.410165,0.418072,0.414081,0.600390,0.589474,0.594882
5,0.0010,12,0.536523,0.030455,0.526210,0.030473,0.536523,0.526821,0.526500,0.526612,...,0.537222,0.573141,0.569048,0.571087,0.410047,0.422892,0.416370,0.597276,0.587560,0.592378
4,0.0010,8,0.531100,0.029148,0.521573,0.029689,0.531100,0.522859,0.521653,0.522077,...,0.532204,0.565321,0.566667,0.565993,0.408257,0.428916,0.418331,0.595000,0.569378,0.581907
11,0.0001,50,0.537161,0.018446,0.517794,0.017509,0.537161,0.519189,0.519805,0.518205,...,0.532474,0.566038,0.595238,0.580271,0.408046,0.342169,0.372215,0.583483,0.622010,0.602131


## Final results — binary classifier

Re-run 5-fold CV at the chosen `(BINARY_LR, BINARY_EPOCHS)` (read off the sweep
heatmap) to produce the final reportable per-fold accuracies and the pooled
(aggregate) classification report.


In [16]:
BINARY_LR = 1e-4
BINARY_EPOCHS = 20
BINARY_FOLDS = 5

In [ ]:
binary_cv = cross_validate(
    binary_data,
    embsize=embsize,
    label_map=BINARY_LABELS,
    lr=BINARY_LR,
    epochs=BINARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=False,
    n_splits=BINARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(binary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(binary_cv.aggregate_classification_report).T.round(3))


## Final results — ternary classifier

Re-run 5-fold CV at the chosen `(TERNARY_LR, TERNARY_EPOCHS)` with inverse-
frequency class weights to produce the final reportable per-fold accuracies and
pooled classification report.


In [ ]:
TERNARY_LR = 1e-3
TERNARY_EPOCHS = 50
TERNARY_FOLDS = 5

In [22]:
ternary_cv = cross_validate(
    ternary_data,
    embsize=embsize,
    label_map=TERNARY_LABELS,
    lr=TERNARY_LR,
    epochs=TERNARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=True,
    n_splits=TERNARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(ternary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(ternary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.471
Fold 2: accuracy=0.419
Fold 3: accuracy=0.450
Fold 4: accuracy=0.542
Fold 5: accuracy=0.539

              precision  recall  f1-score   support
Activation        0.526   0.444     0.482  1213.000
Repression        0.396   0.251     0.307   832.000
None              0.487   0.723     0.582  1018.000
accuracy          0.484   0.484     0.484     0.484
macro avg         0.470   0.473     0.457  3063.000
weighted avg      0.478   0.484     0.468  3063.000


## Save final results

Pickle a `{"binary": CrossValidationResult, "ternary": CrossValidationResult}`
dict to `CV_RESULTS_PATH` for downstream analysis notebooks to load.


In [23]:
CV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "binary": binary_cv,
    "ternary": ternary_cv,
    "embeddings_path": str(EMBEDDINGS_PATH),
    "gene_subset_path": str(GENE_SUBSET_PATH) if GENE_SUBSET_PATH else None,
}
with open(CV_RESULTS_PATH, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved CV results to {CV_RESULTS_PATH}")
print(f"  binary:  lr={binary_cv.config['lr']}, epochs={binary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(binary_cv.fold_accuracies):.3f}")
print(f"  ternary: lr={ternary_cv.config['lr']}, epochs={ternary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(ternary_cv.fold_accuracies):.3f}")


Saved CV results to ../data/results/scgpt_cv_results_nkt.pkl
  binary:  lr=0.0001, epochs=12, mean fold acc=0.696
  ternary: lr=0.0001, epochs=50, mean fold acc=0.485
